# Notebook 05 — Final Load Preparation & Report

**Author:** Section D – Group 8  
**Input:** `data/clean_data.csv`, `reports/cleaning_report.csv`  

**Goal:** Validate the final clean dataset end-to-end, generate summary statistics, review the complete cleaning log, and confirm the data is ready for Tableau or any downstream BI tool.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 30)

df  = pd.read_csv('../data/clean_data.csv')
log = pd.read_csv('../reports/cleaning_report.csv')

print(f'Clean dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Cleaning steps logged: {len(log)}')

## 1. Cleaning Step-by-Step Report

In [ ]:
# Full cleaning log
log

In [ ]:
# Bar chart of rows affected per step
fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(log['Step'], log['RowsAffected'], color='steelblue')
ax.set_title('Cleaning Steps: Rows Affected', fontsize=14)
ax.set_xlabel('Rows Affected')
plt.tight_layout()
plt.show()

## 2. Null Count Verification — All Columns

In [ ]:
# Complete null audit
nulls = df.isnull().sum()
null_df = pd.DataFrame({'Column': nulls.index, 'Null Count': nulls.values})
null_df['Status'] = null_df['Null Count'].apply(lambda x: '✅ Clean' if x == 0 else f'⚠ {x} missing')
null_df

In [ ]:
total_nulls = nulls.sum()
if total_nulls == 0:
    print('🏆 PERFECT CLEAN STATE: 0 missing values across all 0 columns.')
else:
    print(f'⚠ Total remaining nulls: {total_nulls}')

## 3. Final Dataset Shape & Schema

In [ ]:
print(f'Shape: {df.shape}')
print('\nColumn dtypes:')
print(df.dtypes)

In [ ]:
# Preview first 5 rows
df.head()

In [ ]:
# Preview last 5 rows
df.tail()

## 4. Full Descriptive Statistics on Numerical Columns

In [ ]:
df.describe(include='number').T

## 5. Categorical Column Summaries

In [ ]:
for col in ['property_type', 'property_status', 'state']:
    print(f'\n--- {col} ---')
    vc = df[col].value_counts(dropna=False)
    print(vc)

## 6. Key Market Metrics Summary

In [ ]:
metrics = {
    'Total Listings'              : f"{len(df):,}",
    'States Covered'              : f"{df['state'].nunique()}",
    'Cities Covered'              : f"{df['city'].nunique():,}",
    'Median Sale Price'           : f"${df['Sale_Price'].median():,.0f}",
    'Mean Sale Price'             : f"${df['Sale_Price'].mean():,.0f}",
    'Median Living Space'         : f"{df['Living_Space_SqFt'].median():,.0f} sqft",
    'Median $/SqFt'               : f"${df['Price_Per_SqFt'].median():.2f}",
    'Most Common Property Type'   : df['property_type'].mode()[0],
    'Most Common State'           : df['state'].mode()[0],
}
for k, v in metrics.items():
    print(f'{k:35}: {v}')

## 7. Before vs After Cleaning Comparison

In [ ]:
raw_80k = pd.read_csv('../data/raw_extracted_data.csv', low_memory=False)

comparison = pd.DataFrame({
    'Metric': ['Total Rows', 'Total Nulls', 'Median Price', 'LOT Listings Removed'],
    'Before Cleaning': [
        len(raw_80k),
        raw_80k.isnull().sum().sum(),
        f"${pd.to_numeric(raw_80k['price'], errors='coerce').median():,.0f}",
        int((raw_80k['property_type'] == 'LOT').sum())
    ],
    'After Cleaning': [
        len(df),
        df.isnull().sum().sum(),
        f"${df['Sale_Price'].median():,.0f}",
        'N/A (removed)'
    ]
})
comparison

## 8. Price Distribution — Before vs After (Capping Effect)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Sale_Price'].clip(upper=5e6), bins=60, color='salmon', edgecolor='white')
axes[0].set_title('Sale Price (Raw, Clipped for Display)')
axes[0].set_xlabel('Sale Price ($)')

axes[1].hist(df['Sale_Price_Capped'], bins=60, color='teal', edgecolor='white')
axes[1].set_title('Sale Price (Capped at 99th Pct)')
axes[1].set_xlabel('Sale Price ($)')

plt.tight_layout()
plt.show()

## 9. Confirm Data Ready for Tableau

In [ ]:
# Columns available for Tableau
critical = [
    ('Sale_Price',          'Primary price metric'),
    ('Sale_Price_Capped',   'Price capped for visualization'),
    ('Living_Space_SqFt',   'Size of property'),
    ('Living_Space_Capped', 'Capped size'),
    ('Bedrooms',            'Bedroom count'),
    ('Bathrooms',           'Bathroom count'),
    ('Price_Per_SqFt',      'Derived price density metric'),
    ('land_space',          'Plot/land area in sqft'),
    ('latitude',            'Map coordinate'),
    ('longitude',           'Map coordinate'),
    ('state',               'Geographic grouping'),
    ('city',                'Geographic grouping'),
    ('property_type',       'Segmentation dimension'),
    ('property_status',     'Market status'),
    ('agency_name',         'Broker filter'),
]

print(f'{"Column":25} {"Null?":8} {"Purpose"}')
print('-' * 65)
for col, purpose in critical:
    null_n = df[col].isnull().sum() if col in df.columns else 'N/A'
    status = '✅' if null_n == 0 else f'⚠ {null_n}'
    print(f'{col:25} {status:8} {purpose}')

In [ ]:
import os
path = '../data/clean_data.csv'
size_mb = os.path.getsize(path) / 1024 / 1024
print(f'\n✅ Final File: {path}')
print(f'   Rows   : {len(df):,}')
print(f'   Columns: {len(df.columns)}')
print(f'   Size   : {size_mb:.2f} MB')
print(f'\n📊 Ready for Tableau / BI / Visualization.')